In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
def avaliar_modelo_em_teste(
    caminho_csv_teste="database/test_tratado_anova.csv",
    caminho_modelo="models/best_regression_model.pkl",
    caminho_features="models/model_feature_columns.pkl",
):
    print(f"Carregando modelo de: {caminho_modelo}...")
    try:
        model = joblib.load(caminho_modelo)
    except FileNotFoundError:
        print("Erro: Arquivo do modelo não encontrado. Certifique-se de que o arquivo .pkl está no mesmo diretório.")
        return

    print(f"Carregando features do modelo de: {caminho_features}...")
    try:
        feature_columns = joblib.load(caminho_features)
    except FileNotFoundError:
        print("Erro: Arquivo com as features do modelo não encontrado.")
        return

    print(f"Carregando dados de teste de: {caminho_csv_teste}...")
    try:
        df_test = pd.read_csv(caminho_csv_teste)
    except FileNotFoundError:
        print("Erro: Arquivo CSV de teste não encontrado.")
        return

    target = "Preco"
    X_test = df_test.reindex(columns=feature_columns, fill_value=0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    print("\nRealizando predições sem remover linhas...")
    y_pred = model.predict(X_test)

    preco_true = pd.to_numeric(df_test[target], errors="coerce") if target in df_test.columns else pd.Series(np.nan, index=df_test.index)

    df_resultado = pd.DataFrame({
        "Preco": pd.Series(np.rint(preco_true), index=df_test.index).astype("Int64"),
        "Preco_Predito": pd.Series(np.rint(y_pred), index=df_test.index).astype("Int64"),
    })
    df_resultado["Erro"] = (df_resultado["Preco"] - df_resultado["Preco_Predito"]).astype("Int64")
    df_resultado["Erro_Absoluto"] = df_resultado["Erro"].abs().astype("Int64")

    if target in df_test.columns:
        y_true = preco_true
        linhas_metricas = y_true.notna()

        if linhas_metricas.any():
            rmse = np.sqrt(mean_squared_error(y_true[linhas_metricas], y_pred[linhas_metricas]))
            mae = mean_absolute_error(y_true[linhas_metricas], y_pred[linhas_metricas])
            r2 = r2_score(y_true[linhas_metricas], y_pred[linhas_metricas])

            print("\n" + "=" * 30)
            print("MÉTRICAS DE DESEMPENHO NO TESTE")
            print("=" * 30)
            print(f"Linhas no arquivo: {len(df_test)}")
            print(f"Linhas usadas nas métricas: {int(linhas_metricas.sum())}")
            print(f"R² Score: {r2:.4f}")
            print(f"RMSE:     {rmse:.2f}")
            print(f"MAE:      {mae:.2f}")
            print("=" * 30)
        else:
            print("Aviso: coluna Preco existe, mas não há valores válidos para calcular métricas.")
    else:
        print("Aviso: coluna Preco não encontrada. Apenas predições foram geradas.")

    df_resultado.to_csv("teste_com_predicoes.csv", index=False)
    print("\nArquivo 'teste_com_predicoes.csv' gerado com todas as linhas.")

In [3]:
if __name__ == "__main__":
    avaliar_modelo_em_teste("database/test_tratado_anova.csv")

Carregando modelo de: models/best_regression_model.pkl...
Carregando features do modelo de: models/model_feature_columns.pkl...
Carregando dados de teste de: database/test_tratado_anova.csv...

Realizando predições sem remover linhas...

MÉTRICAS DE DESEMPENHO NO TESTE
Linhas no arquivo: 17370
Linhas usadas nas métricas: 9089
R² Score: 0.4348
RMSE:     16549.60
MAE:      6033.94

Arquivo 'teste_com_predicoes.csv' gerado com todas as linhas.
